# ML Masterclass 08: Production Data Engineering & Pipelines

Welcome to Notebook 08. Theoretical ML assumes data is a single, clean $X$ matrix and a $y$ vector. 
In production, data is a messy, flowing pipeline. How you build that pipeline dictates whether your model will generalize or fail gracefully.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"You scaled your features (MinMax) BEFORE doing the Train/Test split. Why is your entire model evaluation now completely invalid due to Mathematical Data Leakage?"*

---

## Table of Contents
1. **Data Leakage:** The silent killer of ML models.
2. **Strict Pipelines:** Using `sklearn.pipeline` to physically prevent leakage.
3. **Extreme Imbalance Strategies:** Why you must NEVER apply SMOTE to your Test set.


## 1. The Mathematics of Data Leakage

Data leakage is when information from outside the training dataset is used to create the model. It causes models to look incredibly accurate during validation but fail wildly in production.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why is scaling BEFORE splitting a form of data leakage?*

**A:** 
Suppose feature A is "Income". You run `MinMaxScaler` on the entire dataset. It finds the global maximum income (say, $1,000,000) and scales everything relative to it. 

Then, you split into Train and Test. 

Your model is now training on data that was mathematically influenced by the $1,000,000 value. But what if that $1,000,000 value only existed in the *Test* set? Your training process just secretly "peaked" at the maximum value of the test set. It learned information about data it wasn't supposed to see. In production, you don't know the future maximum value. 

**Rule:** You must `.fit()` your scalers/imputers ONLY on the train set, and `.transform()` the test set.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# -----------------------------------------------------
# IMPLEMENTATION: The Wrong vs Right Way to Scale
# -----------------------------------------------------

# Generate synthetic data
np.random.seed(42)
X = np.random.randn(1000, 5) * 100 # High variance features
y = (X[:, 0] + X[:, 1] > 0).astype(int) # Arbitrary target

print("--- THE WRONG WAY (Data Leakage) ---")
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X) # Scaled the whole thing!
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_scaled_wrong, y, test_size=0.2, random_state=42)

model_wrong = LogisticRegression()
model_wrong.fit(X_train_w, y_train_w)
print(f"Wrong Approach Accuracy: {accuracy_score(y_test_w, model_wrong.predict(X_test_w)):.4f}\n")

print("--- THE RIGHT WAY (Strict Boundary) ---")
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_right = StandardScaler()
# Fit ONLY on train
X_train_r_scaled = scaler_right.fit_transform(X_train_r)
# Transform test based only on train's mean/variance
X_test_r_scaled = scaler_right.transform(X_test_r)

model_right = LogisticRegression()
model_right.fit(X_train_r_scaled, y_train_r)
print(f"Right Approach Accuracy: {accuracy_score(y_test_r, model_right.predict(X_test_r_scaled)):.4f}")

# The accuracies might look similar on random data, but on complex,
# highly skewed datasets (like wealth or text frequencies), the leakage
# will cause a massive artificial boost to the 'Wrong' score.

## 2. Strict Pipelines (`sklearn.pipeline`)

Handling the "Right Way" manually during cross-validation ($K=5$) is a nightmare. You have to fit the scaler 5 different times on 5 different folds. 

The `Pipeline` object mathematically guarantees that data leakage cannot happen. When you call `cross_val_score(pipeline, X, y)`, the pipeline automatically ensures that inside every single fold, the imputers and scalers are ONLY fitted to the $K-1$ training folds, and only `.transform()` is applied to the validation fold.

## 3. Extreme Imbalance Strategies (SMOTE)

If you have 99% Negative and 1% Positive data, algorithms struggle to learn the decision boundary because the loss function penalty is overwhelmed by the majority class.

**SMOTE (Synthetic Minority Over-sampling Technique)** fixes this by drawing lines between minority points in high-dimensional space and creating fake synthetic points along those lines to balance the dataset 50/50.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why must you NEVER apply SMOTE to your test set?*

**A:** The Test set represents the harsh, biological reality of production. In the real world, only 1% of transactions are fraud. If you run SMOTE on your test set, you test your model in an alternate universe where 50% of transactions are fraud. 

If your model gets 90% accuracy in the fake 50/50 universe, it gives you absolutely no guarantee of how it will perform in the real 1% universe because the **Prior Probabilities** are shifted. SMOTE is strictly a mathematical hack to help the loss function find gradients during training. You must always test on the raw, unadulterated 99/1 split to see if the boundary it found actually works.